# 网格天气现象综合电码生成算法说明

**系统名称**：网格天气现象综合电码生成算法  
**标准依据**：QX/T 740-2024《基于网格预报的城镇预报生成规范 天气现象》  
**架构规范**：国省统筹算法技术规范（三级架构）  
**版本**：v1.0.0

本 Notebook 汇总算法应用场景、原理、实现方式、参数说明与调用示例。  
完整文字版见 `docs/算法说明.md`；集成测试见 `nbs/run_test.ipynb`。

## 0. 路径设置（首次运行必须执行）

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'项目根目录: {PROJECT_ROOT}')
print(f'Python 版本: {sys.version}')

## 一、算法应用场景

| 场景 | 说明 |
|------|------|
| 城镇预报产品生成 | 由网格预报自动生成 12h 时段天气现象表述（如「阴转小雨」「晴间多云」） |
| 预报业务流水线 | 批量处理起报时次（00/12 UTC），覆盖 003~240h，共 20 个 12h 时段 |
| 质控与检验 | 输出 meteva 标准 6D NetCDF，可直接接入检验、对比、统计流程 |
| 替代人工判识 | 按行业标准统一判识/选取/逻辑关系规则，保证产品规范性与可复现性 |

**输入**：逐 3 小时网格预报 NC（R03 / PTYPE03 / TCC / FOG / HAZE / SAND / THUNDER / HAIL）  
**输出**：每个 12h 时段一张网格场，格点值为 5 位整型综合电码 `KAABB`

> 72h 后 FOG/HAZE/SAND/THUNDER/HAIL 常缺失时自动填零，不影响降水类与天空类判识。

## 二、算法原理

依据标准附录 C，对每个 12h 时段、每个格点执行四步诊断：

```
网格预报变量
    │
    ▼
① 判识（附录A）  →  31 种天气现象是否在 12h / 各 3h 时步出现
    │
    ▼
② 选取（第4章）  →  影响级最高的现象 A + 可独立出现的现象 B（或仅 A）
    │
    ▼
③ 逻辑关系（第5章/附录E）→  单一 / 转 / 间 / 伴有，并确定表述先后
    │
    ▼
④ 编码（第6章）  →  5 位综合电码 KAABB
```

### 2.1 判识

| 类别 | 依据 | 要点 |
|------|------|------|
| 降水（表 A.1） | R03 + PTYPE03（+ THUNDER/HAIL） | 12h 分相态累积量分级；雷阵雨/冰雹/雨夹雪/冻雨另有组合条件 |
| 视程障碍（表 A.2） | FOG / HAZE / SAND | 等级场映射到雾/霾/沙尘电码 |
| 天空状况（表 A.3） | TCC | 12h 内至少 1 个 3h 时步满足云量阈值即判为出现 |

每种现象输出 `12h`（窗口是否出现）与 `fine`（4 个 3h 时步是否出现）。

### 2.2 选取

1. **A**：12h 内已出现现象中影响级最高者  
2. **B**：非互斥、有独立出现时间、影响级次高者；无则仅 A  
3. 互斥组（降雨/降雪/雾/霾/沙尘）内只保留影响级最高的一种

### 2.3 逻辑关系

| 关系 | K | 条件概要 |
|------|---|----------|
| 单一 | 0 | 无 B |
| 伴有 | 3 | A、B 分别为雾类与霾类 |
| 间 | 2 | 晴–多云–阴–小雨特定候选对，且精细时步交替出现 |
| 转 | 1 | 其余双现象；按首次出现时间确定先后 |

### 2.4 编码

```
KAABB = K × 10000 + AA × 100 + BB
```

示例：`00101`=多云，`10207`=阴转小雨，`20001`=晴间多云，`35661`=中度霾伴有轻雾

## 三、算法原理的实现方式

```
Level 3  CLI          cli/main.py, cli/runner.py, cli/data_loader.py
Level 2  Plugins      DIA_WeatherPhenomIdentifier / Selector / LogicJudger / Encoder
Level 1  Functions    identify / select / judge / encode / decode
```

| 步骤 | Plugin | 实现要点 |
|------|--------|----------|
| 判识 | `DIA_WeatherPhenomIdentifier` | NumPy 向量化；TCC 自动识别 0~1 / 0~100 |
| 选取 | `DIA_WeatherPhenomSelector` | int8 索引 + 预计算互斥矩阵 `_EXCL` |
| 逻辑 | `DIA_WeatherPhenomLogicJudger` | 伴有/间候选矩阵；首次出现、持续时长、交替判断 |
| 编码 | `DIA_WeatherPhenomEncoder` | LUT 向量化：`K*10000+AA*100+BB` |

`processor.py` 中四个 Plugin 为模块级单例；cli 负责读 NC，src 只做纯内存计算。

## 四、算法参数说明

业务规则见 `resource/weather_config.py`，结构常量见 `resource/data_schema.py`。

### 4.1 输入变量

| 变量 | 含义 | 单位/编码 | 有效时效 |
|------|------|-----------|----------|
| R03 | 3h 累计降水量 | mm | 003~240h |
| PTYPE03 | 降水相态 | 0无/1雨/2雪/3雨夹雪/4冻雨 | 003~240h |
| TCC | 总云量 | % 或 0~1（自动识别） | 003~240h |
| FOG / HAZE / SAND | 雾/霾/沙尘等级 | 见等级映射 | 003~072h |
| THUNDER / HAIL | 雷暴/冰雹标志 | ≥1 为有 | 003~072h |

单时段内存形状：`ndarray[4, lat, lon]`。

### 4.2 主要阈值（摘要）

- **降雨 12h（mm）**：小雨 [0.1,10) … 特大暴雨 ≥250  
- **降雪 12h（mm）**：小雪 [0.1,2.5) … 特大暴雪 ≥30  
- **天空（TCC %）**：晴 ≤30；多云 30~90；阴 ≥90  
- **组合降水阈值**：雷阵雨/冰雹/雨夹雪/冻雨默认 0.1 mm  
- **逻辑关系**：单一=0，转=1，间=2，伴有=3

### 4.3 CLI 运行参数

| 参数 | 默认 | 说明 |
|------|------|------|
| `init_times` | 必填 | 起报时次 `YYYYMMDDHH` |
| `--output-dir` | `./PHENOM` | 输出根目录 |
| `--seg-range START END` | 1~20 | 12h 时段范围 |
| `--data-root` | 环境变量/config | 数据根目录 |
| `--max-seg-workers` | 3 | 时段间并行 |
| `--max-workers` | 4 | 单时段文件并发读 |
| `--stats-only` | off | 只统计不落盘 |

## 五、算法调用

### 5.1 导入 Plugin

In [ ]:
from src import (
    __version__,
    DIA_WeatherPhenomIdentifier,
    DIA_WeatherPhenomSelector,
    DIA_WeatherPhenomLogicJudger,
    DIA_WeatherPhenomEncoder,
)
from src.encoder import decode

print(f'系统版本: v{__version__}')

identifier = DIA_WeatherPhenomIdentifier()
selector   = DIA_WeatherPhenomSelector()
judger     = DIA_WeatherPhenomLogicJudger()
encoder    = DIA_WeatherPhenomEncoder()
print('Plugin 初始化完成')

### 5.2 Plugin 全流程（mock 数据，无需真实 NC）

In [ ]:
import numpy as np

nlat, nlon = 3, 4
shape_fine = (4, nlat, nlon)

# 构造简单场景：阴转小雨（总云量高 + 小雨量级）
data_dict = {
    'R03':     np.full(shape_fine, 2.0, dtype=np.float32),   # 12h 雨量约 8mm → 小雨
    'PTYPE03': np.ones(shape_fine, dtype=np.float32),        # 雨
    'TCC':     np.full(shape_fine, 95.0, dtype=np.float32),  # 阴
    'FOG':     np.zeros(shape_fine, dtype=np.float32),
    'HAZE':    np.zeros(shape_fine, dtype=np.float32),
    'SAND':    np.zeros(shape_fine, dtype=np.float32),
    'THUNDER': np.zeros(shape_fine, dtype=np.float32),
    'HAIL':    np.zeros(shape_fine, dtype=np.float32),
}
# 前两时步无雨，后两时步有雨 → 利于形成「转」
data_dict['R03'][:2] = 0.0

occur                 = identifier.process(data_dict)
idx_A, idx_B          = selector.process(occur)
logic, idx_fa, idx_fb = judger.process(idx_A, idx_B, occur)
result                = encoder.process(idx_fa, idx_fb, logic)

print('综合电码网格:')
print(result)
print('\n单点解读:')
print(decode(int(result[0, 0])))

### 5.3 命令行调用（在终端执行）

```powershell
# 单起报，全部 20 段
python cli/main.py 2026030100

# 只跑 0~72h
python cli/main.py 2026030100 --seg-range 1 6

# 指定数据与输出
python cli/main.py 2026030100 --data-root \\server\share\SCMOC --output-dir D:/output
```

### 5.4 Python 编排调用

In [ ]:
# 完整编排（需真实数据路径时取消注释）
# from cli.runner import run
# run('2026030100', output_dir='./PHENOM', seg_range=(1, 1),
#     max_seg_workers=3, max_workers=4)

# 单时段：加载 + 计算
# from cli.data_loader import load_segment
# from src.processor import run_segment
# data_dict, shape, lat_arr, lon_arr = load_segment('2026030100', seg_idx=1)
# result = run_segment(data_dict, shape, lat_arr, lon_arr,
#                      init_time='2026030100', seg_idx=1, output_dir='./PHENOM')

print('编排调用示例见上方注释；真实跑数请配置 SCMOC_DATA_ROOT 或 --data-root')

### 5.5 相关脚本

| 路径 | 用途 |
|------|------|
| `nbs/run_test.ipynb` | 环境检查、Plugin/集成测试、pytest |
| `nbs/run_bench.py` | 真实数据分阶段耗时 |
| `nbs/plot_analysis.py` | 结果分布分析绘图 |
| `docs/算法说明.md` | 同主题完整文字说明 |